In [26]:
import os
import numpy as np
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

# =========================
# CONFIGURACIÓN
# =========================
load_dotenv()

token = os.getenv("GITHUB_TOKEN")
base_url = os.getenv("OPENAI_BASE_URL")

# =========================
# MODELOS
# =========================
chat_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=token,
    base_url=base_url,
    temperature=0.7
)

summary_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=token,
    base_url=base_url,
    temperature=0
)

embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=token,
    base_url=base_url
)

# =========================
# BASE DE CONOCIMIENTO
# =========================
documento = """
Notebook ideal para programación y gaming:
- Mínimo 16GB RAM
- SSD 512GB
- GPU dedicada (RTX 3050 o superior)
- Procesador i5 o Ryzen 5 hacia arriba

En Chile (PC Factory):
- Gama media: 600.000 - 900.000 CLP
- Gama alta: 900.000 - 1.500.000 CLP

Marcas recomendadas:
- ASUS TUF
- Acer Nitro
- Lenovo Legion
"""

# =========================
# CHUNKING
# =========================
def dividir_texto(texto, size=200):
    return [texto[i:i+size] for i in range(0, len(texto), size)]

chunks = dividir_texto(documento)

# =========================
# CREAR VECTOR DB
# =========================
vector_db = []

for chunk in chunks:
    emb = embeddings_model.embed_query(chunk)
    vector_db.append({
        "texto": chunk,
        "embedding": emb
    })

# =========================
# SIMILITUD COSENO
# =========================
def similitud(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# =========================
# RECUPERAR CONTEXTO (RAG)
# =========================
def obtener_contexto(pregunta, top_k=2):
    emb_pregunta = embeddings_model.embed_query(pregunta)
    
    resultados = []
    
    for item in vector_db:
        score = similitud(emb_pregunta, item["embedding"])
        resultados.append((score, item["texto"]))
    
    resultados.sort(reverse=True)
    
    contexto = "\n".join([texto for _, texto in resultados[:top_k]])
    return contexto

# =========================
# MEMORIA
# =========================
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# =========================
# RESUMEN AUTOMÁTICO
# =========================
def auto_summarize(session_id: str, max_messages=6):
    history = get_session_history(session_id)
    if len(history.messages) > max_messages:
        messages_to_summarize = history.messages[:-2]
        conversation_text = ""
        for msg in messages_to_summarize:
            role = "Usuario" if msg.type == "human" else "Asistente"
            conversation_text += f"{role}: {msg.content}\n"
        
        summary_prompt = f"Resume brevemente requisitos y presupuesto: {conversation_text}"
        summary_response = summary_llm.invoke(summary_prompt)
        
        recent_messages = history.messages[-2:]
        history.clear()
        history.add_ai_message(f"[RESUMEN]: {summary_response.content}")
        history.messages.extend(recent_messages)

# =========================
# PROMPT (CON RAG)
# =========================
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asesor experto de PC Factory Chile. Usa el contexto si es relevante. Responde en CLP."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "Contexto:\n{context}\n\nPregunta: {input}\n\nFormato: Necesidad, Opciones, Evaluación, Recomendación final.")
])

conversation = RunnableWithMessageHistory(
    prompt | chat_llm,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# =========================
# EJECUCIÓN
# =========================
def ejecutar_chatbot():
    session_id = "sesion_clase_ia_final"
    
    inputs = [
        "Hola, necesito un notebook",
        "Lo quiero para programar y jugar",
        "Presupuesto 850.000 pesos"
    ]

    for user_input in inputs:
        print(f"Usuario: {user_input}")
        
        auto_summarize(session_id)
        
        try:
            contexto = obtener_contexto(user_input)

            response = conversation.invoke(
                {
                    "input": user_input,
                    "context": contexto
                },
                config={"configurable": {"session_id": session_id}}
            )
            
            print(f"Bot:\n{response.content}\n")
        
        except Exception as e:
            print(f"Error: {e}")
        
        print("-" * 30)

# =========================
# MAIN
# =========================
if __name__ == "__main__":
    ejecutar_chatbot()

ModuleNotFoundError: No module named 'numpy'